# Line Emission Denoising — Full-Image U-Net (Week-5 pivot)

Per the 2026-06-18 mentor pivot: **full-image** denoising of line-emission velocity channels
(NO patches), using **last week's U-Net** (`DenoisingUNet`, NOT the DDPM), trained channel-by-channel.

- Dataset: line-emission FITS cubes, (201, 600, 600), split at the **cube level** (3 RunID groups
  held out for inference only).
- Channels sampled per cube via the Gaussian sampler (center 100, ~75% in [50,150]).
- Each channel downsampled to **256×256**, per-channel min-max normalised to [0,1].
- Loss: `HybridLoss(0.8, 0.2)` = 0.8·MSE + 0.2·(1−SSIM). Optimizer Adam lr=1e-3.

### Kaggle setup
GPU on, Internet on, *Add Input* → your line-emission Dataset (FITS cubes). The bootstrap finds it
under `/kaggle/input/` and points the split at it.


## 0. Bootstrap (clone repo for src/, locate data)


In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
if ON_KAGGLE:
    REPO='/kaggle/working/EXXA'; PKG=os.path.join(REPO,'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch','line-emission','--depth','1','https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin','line-emission'], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/line-emission'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','pytorch-msssim'], check=True)
    os.chdir(os.path.join(PKG,'notebooks'));  sys.path.insert(0, PKG)
    # find the line-emission data dir under /kaggle/input (contains run_* subfolders)
    hits = glob.glob('/kaggle/input/**/*_dirty.fits', recursive=True)
    DATA_DIR = os.path.dirname(os.path.dirname(hits[0])) if hits else None
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'): os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../data/Line Emission Data'
print('cwd:', os.getcwd(), '| DATA_DIR:', DATA_DIR)

## 1. Imports, device, config


In [ ]:
import time
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.models.unet import DenoisingUNet
from src.utils.losses import HybridLoss
from pytorch_msssim import ssim as ssim_torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
print('device:', device, '| GPUs:', torch.cuda.device_count())

TARGET_SIZE   = 256     # test 256 first; raise to 300 only if VRAM allows
N_SAMPLES     = 50      # channels per cube
EPOCHS        = 30
LR            = 1e-3
BATCH_SIZE    = 8       # auto-reduced below if OOM (do NOT shrink image first)

## 2. Cube-level split (3 RunID groups held out for inference only)


In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(data_dir=DATA_DIR, n_holdout=3,
                                                     val_fraction=0.2, seed=SEED)

## 3. Datasets + DataLoaders (full-image 256x256, per-channel norm)


In [ ]:
train_ds = FITSChannelDataset(train_cubes, n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED)
val_ds   = FITSChannelDataset(val_cubes,   n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED)
print('train items:', len(train_ds), '| val items:', len(val_ds))

def make_loaders(bs):
    nw = 2 if ON_KAGGLE else 0
    return (DataLoader(train_ds, batch_size=bs, shuffle=True,  num_workers=nw, pin_memory=True),
            DataLoader(val_ds,   batch_size=bs, shuffle=False, num_workers=nw, pin_memory=True))
train_loader, val_loader = make_loaders(BATCH_SIZE)

## 4. Model — DenoisingUNet (last week's architecture, full image; t=0, sigmoid output)


In [ ]:
model = DenoisingUNet(str(device))
print('DenoisingUNet params:', f'{sum(p.numel() for p in model.parameters()):,}')
with torch.no_grad():
    o = model(torch.randn(1,1,TARGET_SIZE,TARGET_SIZE,device=device),
              torch.zeros(1,dtype=torch.long,device=device))
print('forward (1,1,%d,%d) -> %s' % (TARGET_SIZE, TARGET_SIZE, tuple(o.shape)))

## 5. Train — 30 epochs, HybridLoss, Adam lr=1e-3

If batch 8 OOMs at 256, we **reduce batch size first** (8→4→2→1), keeping the image at 256 —
per the requirement not to shrink resolution before batch. The batch actually used is printed.


In [ ]:
criterion = HybridLoss(alpha=0.8, beta=0.2)

def build_model_opt():
    m = DenoisingUNet(str(device))
    return m, torch.optim.Adam(m.parameters(), lr=LR)

# OOM-safe batch probe (256px fixed; shrink batch only)
def probe_batch(bs0):
    bs = bs0
    while bs >= 1:
        try:
            m, opt = build_model_opt()
            tl, _ = make_loaders(bs)
            d, c = next(iter(tl)); d, c = d.to(device), c.to(device)
            t = torch.zeros(d.size(0), dtype=torch.long, device=device)
            loss = criterion(torch.sigmoid(m(d, t)), c)[0]
            loss.backward()
            del m, opt, loss, d, c; torch.cuda.empty_cache()
            return bs
        except RuntimeError as e:
            if 'out of memory' not in str(e).lower(): raise
            torch.cuda.empty_cache(); bs //= 2
            print(f'[OOM] reducing batch -> {bs} (image stays {TARGET_SIZE})')
    raise RuntimeError('does not fit even at batch 1')

BATCH_USED = probe_batch(BATCH_SIZE)
print(f'BATCH SIZE USED: {BATCH_USED} at {TARGET_SIZE}x{TARGET_SIZE}')
train_loader, val_loader = make_loaders(BATCH_USED)

In [ ]:
model, optimizer = build_model_opt()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)

CKPT = '../results/checkpoints/unet_line_emission_best.pth'
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
best_val, best_epoch, best_state = float('inf'), -1, None
tr_hist, va_hist = [], []

hdr = f"{'ep':>3} | {'tr_tot':>8} {'tr_mse':>8} {'tr_ssim':>8} | {'va_tot':>8} {'va_mse':>8} {'va_ssim':>8} | {'lr':>8}"
print(hdr); print('-'*len(hdr))

def run_epoch(loader, train):
    model.train(train)
    tot=mse=ssl=0.0; n=0
    torch.set_grad_enabled(train)
    for d, c in loader:
        d, c = d.to(device), c.to(device)
        t = torch.zeros(d.size(0), dtype=torch.long, device=device)
        pred = torch.sigmoid(model(d, t))
        total, m, s = criterion(pred, c)
        if train:
            optimizer.zero_grad(); total.backward(); optimizer.step()
        bs=d.size(0); tot+=total.item()*bs; mse+=m.item()*bs; ssl+=s.item()*bs; n+=bs
    torch.set_grad_enabled(True)
    return tot/n, mse/n, ssl/n

for ep in range(1, EPOCHS+1):
    t0=time.time()
    tr=run_epoch(train_loader, True)
    va=run_epoch(val_loader, False)
    scheduler.step(va[0]); tr_hist.append(tr[0]); va_hist.append(va[0])
    mark=''
    if va[0] < best_val:
        best_val, best_epoch = va[0], ep
        best_state={k:v.clone() for k,v in model.state_dict().items()}
        torch.save({'epoch':ep,'model_state_dict':best_state,'val_loss':best_val,
                    'alpha':0.8,'beta':0.2,'arch':'DenoisingUNet','target_size':TARGET_SIZE,
                    'batch_size':BATCH_USED}, CKPT)
        mark=' *best'
    print(f"{ep:>3} | {tr[0]:>8.4f} {tr[1]:>8.4f} {tr[2]:>8.4f} | "
          f"{va[0]:>8.4f} {va[1]:>8.4f} {va[2]:>8.4f} | {optimizer.param_groups[0]['lr']:>8.1e}"
          f"  {time.time()-t0:.0f}s{mark}")

print(f'\nbest val total {best_val:.4f} @ epoch {best_epoch}; checkpoint -> {CKPT}')
if best_state: model.load_state_dict(best_state)

## 6. Loss curve


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(range(1,len(tr_hist)+1), tr_hist, marker='o', ms=3, label='train')
plt.plot(range(1,len(va_hist)+1), va_hist, marker='s', ms=3, label='val')
plt.axvline(best_epoch, color='gray', ls=':'); plt.scatter([best_epoch],[best_val],color='#E8715A',zorder=5)
plt.xlabel('epoch'); plt.ylabel('Hybrid loss'); plt.title(f'Line-emission U-Net ({TARGET_SIZE}px, batch {BATCH_USED})')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/unet_line_emission_loss.png', dpi=140); plt.show()

## 7. Validation metrics — PSNR / SSIM / MSE


In [ ]:
model.eval()
psnrs, ssims, mses = [], [], []
with torch.no_grad():
    for d, c in val_loader:
        d, c = d.to(device), c.to(device)
        t = torch.zeros(d.size(0), dtype=torch.long, device=device)
        pred = torch.sigmoid(model(d, t)).clamp(0,1)
        mse = torch.mean((pred-c)**2, dim=(1,2,3))
        psnrs += (10*torch.log10(1.0/torch.clamp(mse,min=1e-10))).cpu().tolist()
        ssims += ssim_torch(pred, c, data_range=1.0, size_average=False).cpu().tolist()
        mses  += mse.cpu().tolist()
print(f'Validation ({len(mses)} channels):  PSNR {np.mean(psnrs):.4f} dB | '
      f'SSIM {np.mean(ssims):.4f} | MSE {np.mean(mses):.6f}')

## 8. Visualize 5 random validation channels — dirty | denoised | clean


In [ ]:
import random
idxs = random.Random(SEED).sample(range(len(val_ds)), 5)
fig, ax = plt.subplots(5, 3, figsize=(10, 16))
cols=['dirty','U-Net denoised','clean GT']
for k, t in enumerate(cols): ax[0,k].set_title(t, fontweight='bold')
model.eval()
with torch.no_grad():
    for r, ix in enumerate(idxs):
        d, c = val_ds[ix]
        t0 = torch.zeros(1, dtype=torch.long, device=device)
        pred = torch.sigmoid(model(d[None].to(device), t0))[0,0].cpu().numpy()
        ci, ch = val_ds.index[ix]
        for col, im in enumerate([d[0].numpy(), pred, c[0].numpy()]):
            ax[r,col].imshow(np.clip(im,0,1), cmap='inferno'); ax[r,col].axis('off')
        ax[r,0].set_ylabel(f'{val_ds.cube_paths[ci][2]}\nch {ch}', fontsize=8)
fig.suptitle('Line-emission U-Net — validation channels', fontweight='bold', y=0.995)
plt.tight_layout()
os.makedirs('../experiments', exist_ok=True)
plt.savefig('../experiments/line_emission_unet_comparison.png', dpi=140); plt.show()
print('saved -> experiments/line_emission_unet_comparison.png')